# Day 14: Exploratory Data Analysis & Feature Validation

This notebook performs quality audits, multi-collinearity checks, and tree-based feature importance on the engineered feature store.

In [1]:
import polars as pl
import numpy as np
from src.features.feature_validation import audit_feature_quality, analyze_correlations_and_redundancy, compute_feature_importances, select_features

train_df = pl.read_parquet("data/features/train_features.parquet")
print("Train Matrix Shape:", train_df.shape)

In [2]:
# 1. Feature Quality Audit
audit = audit_feature_quality(train_df)
print("Audit is_clean:", audit["is_clean"])
print("Infinite columns:", audit["infinite_columns"])
print("Columns with nulls:", list(audit["missing_summary"].keys()))

In [3]:
# 2. Correlation with Target and Multi-Collinearity
target_corrs, collinear = analyze_correlations_and_redundancy(train_df)
print("Top target correlations:")
for k, v in list(target_corrs.items())[:6]:
    print(f"  {k:<30}: {v:.4f}")
print(f"
Detected {len(collinear)} multi-collinear pairs (|r| >= 0.90)")

In [4]:
# 3. Tree-Based Importance & Selection
importances = compute_feature_importances(train_df)
selection = select_features(train_df, importances, collinear, top_k=18)
print(f"Selected {selection['selected_feature_count']} features.")
print("Top 5 predictive features:")
for item in selection["top_predictive_features"]:
    print(f"  {item['feature']:<30}: {item['importance']:.4f}")